In [18]:
!pip -q install -U transformers accelerate peft bitsandbytes optuna
import os, shutil, gc, torch, optuna
from huggingface_hub import HfApi, hf_hub_download, login
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training)
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from datasets import load_dataset, concatenate_datasets
import json
from collections import Counter
from torch.utils.data import Dataset

In [9]:
REPO_ID   = "eduhuemar001/tinyllama-german-sentiment-4bit-tuning"
STUDY_DB  = "optuna_study.db"
STUDY_KEY = "optuna/optuna_study.db"
STUDY_NAME= "qlora_tinyllama_study"
N_TRIALS  = 10

login()
api = HfApi()

In [10]:
try:
    fp = hf_hub_download(REPO_ID, STUDY_KEY, local_dir=".", local_dir_use_symlinks=False)
    if os.path.basename(fp) != STUDY_DB:
        shutil.copy(fp, STUDY_DB)
    print("Loaded existing Optuna DB from Hub.")
except Exception:
    print("No existing Optuna DB on Hub; starting fresh.")

storage = f"sqlite:///{STUDY_DB}"

No existing Optuna DB on Hub; starting fresh.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


In [11]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer once
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Decide compute dtype (T4: fp16; A100/H100: bf16)
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

In [12]:
def build_model(lora_r=8, lora_alpha=16, lora_dropout=0.05):
    # Load 4-bit base
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=quant_config,
    )
    # Prepare for k-bit LoRA training
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False

    # Your LoRA config (targets as in your code; expand if you like)
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "v_proj"],   # your original choice
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, lora_config)
    return model

In [17]:
# === CONSTANTS ===
HF_DATASET_REPO_ID = "eduhuemar001/dataset-sentiment-4bit-train-tuning"
HF_DATASET_REPO_ID_TEST = "eduhuemar001/dataset-sentiment-4bit-test-tuning"
DATASET_FILE = "current_dataset.json"
LOCAL_DATASET_PATH = f"/content/{DATASET_FILE}"
REMOTE_DATASET_PATH = f"{DATASET_FILE}"

HF_DATASET_REPO_GERMEVAL = "eduhuemar001/sentiment-GermEval2017"
HF_DATASET_REPO_SB10K = "eduhuemar001/sentiment-SB10k"
HF_DATASET_REPO_HOLIDAYCHECK = "eduhuemar001/sentiment-holidaycheck-balanced"

TRAIN_PATH = "/content/train_dataset.json"
EVAL_PATH  = "/content/eval_dataset.json"   # <-- added
TEST_PATH = "/content/test_dataset.json"
EVAL_JSON = "eval_dataset.json"             # <-- added

api = HfApi()

# === Check if dataset already exists ===
try:
    files = list_repo_files(HF_DATASET_REPO_ID, repo_type="dataset")
    dataset_exists = REMOTE_DATASET_PATH in files
except Exception as e:
    print(f"Repo does not exist or could not be listed: {e}")
    files = []
    dataset_exists = False

if dataset_exists:
    print("Loading existing dataset from Hugging Face Hub...")

    dataset_path = hf_hub_download(
        repo_id=HF_DATASET_REPO_ID,
        filename=REMOTE_DATASET_PATH,
        repo_type="dataset",
        local_dir="/content",
        local_dir_use_symlinks=False
    )

    # Read JSON lines (not a single JSON object)
    with open(dataset_path, "r", encoding="utf-8") as f:
        dataset = [json.loads(line) for line in f if line.strip()]

    # Use persistent splits from Hub
    train_path_hub = hf_hub_download(
        repo_id=HF_DATASET_REPO_ID,
        filename="train_dataset.json",
        repo_type="dataset",
        local_dir="/content",
        local_dir_use_symlinks=False
    )
    eval_path_hub = hf_hub_download(
        repo_id=HF_DATASET_REPO_ID,
        filename=EVAL_JSON,
        repo_type="dataset",
        local_dir="/content",
        local_dir_use_symlinks=False
    )
    with open(train_path_hub, "r", encoding="utf-8") as f:
        dataset = [json.loads(line) for line in f if line.strip()]             # keep 'dataset' as your train list
    with open(eval_path_hub, "r", encoding="utf-8") as f:
        dataset_eval = [json.loads(line) for line in f if line.strip()]

else:
    print("No existing dataset found. Loading and balancing from sentiment datasets...")

    # === Load datasets ===
    ds_germeval = load_dataset(HF_DATASET_REPO_GERMEVAL, split="train")
    ds_sb10k = load_dataset(HF_DATASET_REPO_SB10K, split="train")
    ds_holidaycheck = load_dataset(HF_DATASET_REPO_HOLIDAYCHECK, split="train")

    # === Count sentiment samples ===
    combined_existing = concatenate_datasets([ds_germeval, ds_sb10k])
    counts = Counter(combined_existing["sentiment"])
    print("Before balancing:", dict(counts))

    max_count = max(counts.values())
    needed_pos = max_count - counts.get("positive", 0)
    needed_neg = max_count - counts.get("negative", 0)

    # === Add positive/negative from HolidayCheck ===
    hc_pos = ds_holidaycheck.filter(lambda x: x["sentiment"] == "positive").shuffle(seed=42).select(range(needed_pos))
    hc_neg = ds_holidaycheck.filter(lambda x: x["sentiment"] == "negative").shuffle(seed=42).select(range(needed_neg))

    # === Combine and shuffle ===
    combined_ds = concatenate_datasets([ds_germeval, ds_sb10k, hc_pos, hc_neg]).shuffle(seed=42)
    final_counts = Counter(combined_ds["sentiment"])
    print("After balancing:", dict(final_counts))

    # === Train/test split ===
    split = combined_ds.train_test_split(test_size=0.2, seed=42)
    train_ds = split["train"]
    test_ds = split["test"]

    # --- ALSO create eval from train (≈10% of total -> 0.125 of train) ---
    split_tr = train_ds.train_test_split(test_size=0.125, seed=42)
    train_ds = split_tr["train"]
    eval_ds  = split_tr["test"]

    print("\nTrain counts:", Counter(train_ds["sentiment"]))
    print("Eval counts:", Counter(eval_ds["sentiment"]))
    print("Test counts:", Counter(test_ds["sentiment"]))

    # === Save function ===
    def export_dataset(ds, path):
        json_data = []
        with open(path, "w", encoding="utf-8") as f:
            for example in ds:
                if example["review_text"] and example["sentiment"]:
                    item = {
                        "review_text": example["review_text"].strip(),
                        "sentiment": example["sentiment"].strip()
                    }
                    json.dump(item, f, ensure_ascii=False)
                    f.write("\n")
                    json_data.append(item)
        return json_data

    # === Save train/eval/test to disk ===
    print("Saving datasets to disk...")
    dataset       = export_dataset(train_ds, TRAIN_PATH)  # keep variable name 'dataset' for train
    dataset_eval  = export_dataset(eval_ds,  EVAL_PATH)   # <-- added
    export_dataset(test_ds, TEST_PATH)

    # === Upload to Hugging Face Hub ===
    print("Uploading datasets to Hugging Face Hub...")

    # Upload train + eval split
    api.create_repo(repo_id=HF_DATASET_REPO_ID, repo_type="dataset", exist_ok=True, token=True)
    api.upload_file(
        path_or_fileobj=TRAIN_PATH,
        path_in_repo="train_dataset.json",
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Upload sentiment train split"
    )
    api.upload_file(
        path_or_fileobj=EVAL_PATH,
        path_in_repo=EVAL_JSON,
        repo_id=HF_DATASET_REPO_ID,
        repo_type="dataset",
        commit_message="Upload sentiment eval split"
    )

    # Upload test split
    api.create_repo(repo_id=HF_DATASET_REPO_ID_TEST, repo_type="dataset", exist_ok=True, token=True)
    api.upload_file(
        path_or_fileobj=TEST_PATH,
        path_in_repo="test_dataset.json",
        repo_id=HF_DATASET_REPO_ID_TEST,
        repo_type="dataset",
        commit_message="Upload sentiment test split"
    )

    print("Upload complete.")

Repo does not exist or could not be listed: name 'list_repo_files' is not defined
No existing dataset found. Loading and balancing from sentiment datasets...
Before balancing: {'neutral': 17743, 'positive': 2409, 'negative': 6022}


Filter:   0%|          | 0/1671738 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1671738 [00:00<?, ? examples/s]

After balancing: {'negative': 17743, 'positive': 17743, 'neutral': 17743}

Train counts: Counter({'negative': 12478, 'positive': 12404, 'neutral': 12378})
Eval counts: Counter({'negative': 1786, 'positive': 1781, 'neutral': 1756})
Test counts: Counter({'neutral': 3609, 'positive': 3558, 'negative': 3479})
Saving datasets to disk...
Uploading datasets to Hugging Face Hub...


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/train_dataset.json           :   5%|4         |  824kB / 18.0MB            

Upload complete.


In [19]:
class SentimentDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.data = []

        # Fixed parts of the prompt
        self.instruction_prefix = (
            "### Instruction:\n"
            "Klassifiziere die Stimmung der folgenden Bewertung als 'positiv', 'neutral' oder 'negativ'.\n\n"
            "### Bewertung:\n"
        )
        self.answer_prefix = "\n\n### Antwort:\n"

        for item in dataset:
            review_text = item["review_text"].strip()
            sentiment = item["sentiment"].strip()

            # Tokenize fixed parts
            prefix_tokens = tokenizer(self.instruction_prefix, add_special_tokens=False)["input_ids"]
            review_tokens = tokenizer(review_text, add_special_tokens=False)["input_ids"]
            answer_prefix_tokens = tokenizer(self.answer_prefix, add_special_tokens=False)["input_ids"]
            label_tokens = tokenizer(sentiment, add_special_tokens=False)["input_ids"]

            # Calculate how many review tokens fit
            reserved = len(prefix_tokens) + len(answer_prefix_tokens) + len(label_tokens)
            max_review_len = self.max_length - reserved
            if max_review_len <= 0:
                continue  # skip if too long

            review_tokens = review_tokens[:max_review_len]

            # Combine all
            input_ids = prefix_tokens + review_tokens + answer_prefix_tokens + label_tokens
            labels = [-100] * (len(prefix_tokens) + len(review_tokens) + len(answer_prefix_tokens)) + label_tokens

            # Pad
            pad_len = self.max_length - len(input_ids)
            input_ids += [tokenizer.pad_token_id] * pad_len
            labels += [-100] * pad_len

            # Sanity check
            if len(input_ids) != self.max_length or len(labels) != self.max_length:
                continue

            self.data.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long)
            })

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# Build train/eval datasets (train uses 'dataset' as before; eval uses 'dataset_eval')
train_dataset = SentimentDataset(dataset, tokenizer, max_length=256)
eval_dataset  = SentimentDataset(dataset_eval, tokenizer, max_length=256)

Token indices sequence length is longer than the specified maximum sequence length for this model (2079 > 2048). Running this sequence through the model will result in indexing errors


In [ ]:
def run_single_trial_and_get_eval_loss(**hp):
    model = build_model(
        lora_r=hp["lora_r"],
        lora_alpha=hp["lora_alpha"],
        lora_dropout=hp["lora_dropout"]
    )
    model.gradient_checkpointing_enable()

    args = TrainingArguments(
        output_dir=f"./runs/trial_tmp",
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=8,
        max_steps=800,                              # short budget for HPO
        learning_rate=hp["learning_rate"],
        warmup_ratio=hp["warmup_ratio"],
        weight_decay=hp["weight_decay"],
        lr_scheduler_type=hp["lr_scheduler_type"],
        evaluation_strategy="steps",
        eval_steps=150,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        fp16=not USE_BF16,
        bf16=USE_BF16,
        optim="paged_adamw_8bit",
        dataloader_num_workers=2,
    )

    trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=args,
        train_dataset=train_tokenized,
        eval_dataset=dev_tokenized,
        data_collator=default_data_collator,
    )
    trainer.train()
    metrics = trainer.evaluate()
    eval_loss = float(metrics["eval_loss"])
    # cleanup
    del trainer, model
    torch.cuda.empty_cache(); gc.collect()
    return eval_loss

In [ ]:
def objective(trial: optuna.Trial):
    r  = trial.suggest_categorical("lora_r", [8, 16])
    alpha = trial.suggest_categorical("lora_alpha", [2*r, 3*r])       # scaled to r
    hp = {
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 3e-4, log=True),
        "warmup_ratio":  trial.suggest_float("warmup_ratio", 0.03, 0.08),
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["linear","cosine"]),
        "lora_dropout":  trial.suggest_float("lora_dropout", 0.0, 0.05),
        "weight_decay":  trial.suggest_float("weight_decay", 0.0, 0.05),
        "lora_r":        r,
        "lora_alpha":    alpha,
    }
    # mild LR cap if r is high
    if r == 16 and hp["learning_rate"] > 2.5e-4:
        hp["learning_rate"] = 2.5e-4
    return run_single_trial_and_get_eval_loss(**hp)

In [ ]:
try:
    study = optuna.load_study(study_name=STUDY_NAME, storage=storage)
    print("Resumed existing study.")
except KeyError:
    study = optuna.create_study(
        study_name=STUDY_NAME,
        storage=storage,
        direction="minimize",
        sampler=TPESampler(multivariate=True, group=True, n_startup_trials=3, seed=42),
        pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=300),
    )
    print("Created new study.")

In [ ]:
def push_db_callback(study: optuna.Study, trial: optuna.trial.FrozenTrial):
    api.upload_file(
        path_or_fileobj=STUDY_DB,
        path_in_repo=STUDY_KEY,
        repo_id=REPO_ID,
        repo_type="model",  # change to "dataset" if your repo is a dataset repo
        commit_message=f"[auto] sync DB after trial {trial.number} ({trial.state})"
    )
    print(f"Synced Optuna DB to Hub after trial {trial.number}.")

In [ ]:
study.optimize(objective, n_trials=N_TRIALS, callbacks=[push_db_callback], show_progress_bar=True)

In [ ]:
print("Best value:", study.best_value)
print("Best params:", study.best_params)

# Final sync
push_db_callback(study, study.best_trial)

In [ ]:
"""
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 4-bit quantization config (bnb)
# Tip: use float16 on T4/V100; use bfloat16 on A100/H100
compute_dtype = torch.float16  # change to torch.bfloat16 if you're on A100/H100
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # best quality for 4-bit
    bnb_4bit_use_double_quant=True,   # nested quantization saves memory
    bnb_4bit_compute_dtype=compute_dtype
)

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quant_config,
)

# Prepare for k-bit LoRA training (enables input grads, fixes layer norms, etc.)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False  # important for training (avoids warnings with grad_checkpointing)

# Define LoRA configuration (you can also target more modules for better quality)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # common minimal set; consider adding "k_proj","o_proj","gate_proj","up_proj","down_proj"
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to the quantized model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
